# Verify Analytical Value Function

**Goal**: Ensure `get_exact_value()` matches Monte Carlo simulation.

This notebook tests ONE (NUM_AGENTS, PICKER_PENALTY) combination for SLURM parallelization.

In [ ]:
# Parameters (can be overridden by papermill)
NUM_AGENTS = 4
PICKER_PENALTY = 0.25

# Fixed settings
WIDTH = 9
HEIGHT = 9
NUM_APPLES = 40
GAMMA = 0.99

# Monte Carlo settings
MC_SAMPLES = 2000
MC_DEPTH = 1000

# Verification settings
NUM_COMPARISONS = 10
ERROR_THRESHOLD = 1.0

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys
sys.path.append("../../")

from tadd_helpers.env_functions import State
from teleport_dynamic.rewards_decentralized import make_picker_penalty_reward
from teleport_dynamic.analytical import get_exact_value, calculate_stream_values
from teleport_dynamic.orchard_generator import init_fixed_apples, teleport_agent

np.random.seed(42)
print(f"Verifying: NUM_AGENTS={NUM_AGENTS}, PICKER_PENALTY={PICKER_PENALTY}")

import psutil
import os

def get_ram_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024

In [ ]:
def randomize_state(state, num_apples):
    """Randomize both agent positions AND apple positions."""
    H, W = state.H, state.L
    num_agents = len(state._agents)
    
    # Randomize agents
    for i in range(num_agents):
        r = np.random.randint(0, H)
        c = np.random.randint(0, W)
        state.set_agent_position(i, np.array([r, c]))
    
    # Randomize apples
    state.apples = np.zeros((H, W), dtype=np.float32)
    apple_positions = np.random.choice(H * W, size=num_apples, replace=False)
    for pos in apple_positions:
        r, c = pos // W, pos % W
        state.apples[r, c] = 1

def run_monte_carlo(
    initial_state: State, 
    initial_actor_idx: int, 
    self_agent_idx: int, 
    reward_func, 
    gamma: float, 
    depth: int, 
    num_samples: int
) -> float:
    """Monte Carlo estimate of V(s) for a specific agent."""
    total_return = 0.0
    num_agents = len(initial_state._agents)
    
    for _ in range(num_samples):
        episode_state = initial_state.copy()
        current_actor = initial_actor_idx
        
        episode_ret = 0.0
        gamma_pow = 1.0
        print(f"Starting episode {_+1}/{num_samples}")
        for _ in range(depth):
            if _ % 100 == 0:
                print(f"RAM: {get_ram_mb():.1f} MB")
            rewards = reward_func(episode_state, current_actor)
            r = rewards[self_agent_idx]
            episode_ret += gamma_pow * r
            
            teleport_agent(episode_state, current_actor)
            current_actor = np.random.randint(0, num_agents)
            
            gamma_pow *= gamma
            if gamma_pow < 1e-9:
                break
                
        total_return += episode_ret
        
    return total_return / num_samples

## 1. Reward Function Sanity Check

In [ ]:
# Verify reward function produces expected values
print("=== Reward Function Check ===")

dummy_state = init_fixed_apples(3, 3, NUM_AGENTS, 1)
dummy_state.apples[0, 0] = 1
dummy_state.set_agent_position(0, np.array([0, 0]))
for i in range(1, NUM_AGENTS):
    dummy_state.set_agent_position(i, np.array([1, 1]))

reward_func = make_picker_penalty_reward(PICKER_PENALTY)
rewards = reward_func(dummy_state, 0)

picker_r = rewards[0]
other_r = rewards[1] if NUM_AGENTS > 1 else 0
total = picker_r + (NUM_AGENTS - 1) * other_r

expected_picker = PICKER_PENALTY
expected_other = (1 - PICKER_PENALTY) / (NUM_AGENTS - 1) if NUM_AGENTS > 1 else 0

print(f"  picker_reward: {picker_r:.4f} (expected: {expected_picker:.4f})")
print(f"  other_reward:  {other_r:.4f} (expected: {expected_other:.4f})")
print(f"  total:         {total:.4f} (expected: 1.0)")

assert abs(picker_r - expected_picker) < 1e-6, "Picker reward mismatch!"
assert abs(other_r - expected_other) < 1e-6, "Other reward mismatch!"
assert abs(total - 1.0) < 1e-6, "Total reward != 1.0!"
print("  ✓ Reward function correct")

## 2. Stream Values Check

In [ ]:
print("=== Stream Values Check ===")
p_hit = NUM_APPLES / (WIDTH * HEIGHT)
print(f"  p_hit = {p_hit:.4f}")

rho_self = PICKER_PENALTY
rho_other = (1 - PICKER_PENALTY) / (NUM_AGENTS - 1) if NUM_AGENTS > 1 else 0

v_rand_s, v_off_s, v_on_s = calculate_stream_values(rho_self, GAMMA, NUM_AGENTS, p_hit)
v_rand_o, v_off_o, v_on_o = calculate_stream_values(rho_other, GAMMA, NUM_AGENTS, p_hit)

print(f"  rho_self = {rho_self:.4f}")
print(f"  rho_other = {rho_other:.4f}")
print(f"  v_rand_self = {v_rand_s:.4f}")
print(f"  v_rand_other = {v_rand_o:.4f}")
print(f"  v_off_self = {v_off_s:.4f}")
print(f"  v_on_self = {v_on_s:.4f}")

## 3. Monte Carlo vs Analytical Comparison

In [ ]:
print(f"=== Running {NUM_COMPARISONS} Comparisons ===")
print(f"{'#':<4} | {'State':<12} | {'V_exact':<12} | {'V_mc':<12} | {'Diff':<10} | {'Error %':<10} | Status")
print("-" * 85)

base_state = init_fixed_apples(WIDTH, HEIGHT, NUM_AGENTS, NUM_APPLES)
reward_func = make_picker_penalty_reward(PICKER_PENALTY)

results = []

for comp_idx in range(NUM_COMPARISONS):
    print(f"Comparison {comp_idx + 1}/{NUM_COMPARISONS}")
    test_state = base_state.copy()
    randomize_state(test_state, NUM_APPLES)
    actor_idx = np.random.randint(0, NUM_AGENTS)
    self_idx = 0
    
    # Check state type
    actor_pos = test_state.agent_position(actor_idx)
    is_hit = test_state.apples[actor_pos[0], actor_pos[1]] > 0
    state_type = "HIT" if is_hit else "MISS"
    
    # Analytical
    v_exact = get_exact_value(test_state, actor_idx, self_idx, reward_func, GAMMA)
    
    # Monte Carlo
    v_mc = run_monte_carlo(test_state, actor_idx, self_idx, reward_func, GAMMA, MC_DEPTH, MC_SAMPLES)
    
    # Error
    abs_diff = abs(v_exact - v_mc)
    if abs(v_exact) > 0.1:
        pct_err = (abs_diff / abs(v_exact)) * 100
    else:
        pct_err = abs_diff * 100
    
    # Signed error (bias) - MC minus Exact
    signed_diff = v_mc - v_exact
    if abs(v_exact) > 0.1:
        signed_pct = (signed_diff / abs(v_exact)) * 100
    else:
        signed_pct = signed_diff * 100
    
    status = "✓" if pct_err < ERROR_THRESHOLD else "✗"
    
    print(f"{comp_idx:<4} | {state_type:<12} | {v_exact:<12.4f} | {v_mc:<12.4f} | {abs_diff:<10.4f} | {pct_err:<10.2f} | {status}")
    
    results.append({
        'comparison_idx': comp_idx,
        'state_type': state_type,
        'v_exact': v_exact,
        'v_mc': v_mc,
        'abs_diff': abs_diff,
        'pct_error': pct_err,
        'signed_pct': signed_pct
    })

results_df = pd.DataFrame(results)

## 4. Summary

In [ ]:
mean_err = results_df['pct_error'].mean()
mean_bias = results_df['signed_pct'].mean()
max_err = results_df['pct_error'].max()
num_failures = (results_df['pct_error'] >= ERROR_THRESHOLD).sum()

print(f"\n=== Summary ===")
print(f"  NUM_AGENTS: {NUM_AGENTS}")
print(f"  PICKER_PENALTY: {PICKER_PENALTY}")
print(f"  Mean Error: {mean_err:.4f}%")
print(f"  Max Error: {max_err:.4f}%")
print(f"  Mean Bias (signed): {mean_bias:+.4f}%")
print(f"  Failures: {num_failures}/{NUM_COMPARISONS}")

if num_failures == 0:
    print(f"\n✓ PASSED: Analytical matches MC within {ERROR_THRESHOLD}%")
else:
    print(f"\n✗ FAILED: {num_failures} comparisons exceeded {ERROR_THRESHOLD}% error")

In [ ]:
# Save results for aggregation
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

results_df['num_agents'] = NUM_AGENTS
results_df['picker_penalty'] = PICKER_PENALTY
results_df['mc_samples'] = MC_SAMPLES
results_df['mc_depth'] = MC_DEPTH
output_file = output_dir / f"verify_n{NUM_AGENTS}_p{PICKER_PENALTY}_d{MC_DEPTH}_s{MC_SAMPLES}.csv"
results_df.to_csv(output_file, index=False)
print(f"\nResults saved to: {output_file}")